In [1]:
import pandas as pd
data = pd.read_csv('exoTrain.csv')
print(data.head())
print(data.shape)

   LABEL   FLUX.1   FLUX.2   FLUX.3   FLUX.4   FLUX.5   FLUX.6  FLUX.7  \
0      2    93.85    83.81    20.10   -26.98   -39.56  -124.71 -135.18   
1      2   -38.88   -33.83   -58.54   -40.09   -79.31   -72.81  -86.55   
2      2   532.64   535.92   513.73   496.92   456.45   466.00  464.50   
3      2   326.52   347.39   302.35   298.13   317.74   312.70  322.33   
4      2 -1107.21 -1112.59 -1118.95 -1095.10 -1057.55 -1034.48 -998.34   

    FLUX.8  FLUX.9  ...  FLUX.3188  FLUX.3189  FLUX.3190  FLUX.3191  \
0   -96.27  -79.89  ...     -78.07    -102.15    -102.15      25.13   
1   -85.33  -83.97  ...      -3.28     -32.21     -32.21     -24.89   
2   486.39  436.56  ...     -71.69      13.31      13.31     -29.89   
3   311.31  312.42  ...       5.71      -3.73      -3.73      30.05   
4 -1022.71 -989.57  ...    -594.37    -401.66    -401.66    -357.24   

   FLUX.3192  FLUX.3193  FLUX.3194  FLUX.3195  FLUX.3196  FLUX.3197  
0      48.57      92.54      39.32      61.42       5.08  

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
X= data.drop('LABEL', axis=1)
y = data['LABEL']
X = torch.tensor(X.values, dtype=torch.float32)
y = torch.tensor(y.values, dtype=torch.long)
X_train, X_val, y_train, y_val = train_test_split( X, y, test_size=0.2, random_state=42 )
class MLP(nn.Module):
    def __init__(self, input_size):
        super(MLP, self).__init__()
        
        self.model = nn.Sequential(
            nn.Linear(input_size, 128), #learns weight 
            nn.ReLU(), #adds non linearity - negative input values to zero while positive values unchanged
            nn.Dropout(0.3),#prevents overfitting
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(64, 3)  # 3 classes
        )

    def forward(self, x):
        return self.model(x)

input_size = X_train.shape[1]
model = MLP(input_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 20#training loop
for epoch in range(epochs):
    
    model.train()
    
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # validation
    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = criterion(val_outputs, y_val)
    
    print(f"Epoch {epoch+1}, Train Loss: {loss.item()}, Val Loss: {val_loss.item()}")

    _, preds = torch.max(val_outputs, 1)
accuracy = (preds == y_val).float().mean()

print("Validation Accuracy:", accuracy.item())

Epoch 1, Train Loss: 388.0487365722656, Val Loss: 59.233638763427734
Epoch 2, Train Loss: 91.72525787353516, Val Loss: 48.866844177246094
Epoch 3, Train Loss: 50.40524673461914, Val Loss: 29.5341796875
Epoch 4, Train Loss: 24.731731414794922, Val Loss: 9.009452819824219
Epoch 5, Train Loss: 9.459275245666504, Val Loss: 8.569504737854004
Epoch 6, Train Loss: 7.459275722503662, Val Loss: 10.837881088256836
Epoch 7, Train Loss: 4.395975589752197, Val Loss: 13.51069164276123
Epoch 8, Train Loss: 4.935059070587158, Val Loss: 16.317914962768555
Epoch 9, Train Loss: 4.3917317390441895, Val Loss: 18.94581413269043
Epoch 10, Train Loss: 4.548995018005371, Val Loss: 21.402433395385742
Epoch 11, Train Loss: 4.624955177307129, Val Loss: 23.64339828491211
Epoch 12, Train Loss: 4.4796881675720215, Val Loss: 25.734968185424805
Epoch 13, Train Loss: 4.439363956451416, Val Loss: 27.64763832092285
Epoch 14, Train Loss: 4.0800957679748535, Val Loss: 29.156726837158203
Epoch 15, Train Loss: 4.628279685974

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
X= data.drop('LABEL', axis=1)
y = data['LABEL']
X = torch.tensor(X.values, dtype=torch.float32)
y = torch.tensor(y.values, dtype=torch.long)
X_train, X_val, y_train, y_val = train_test_split( X, y, test_size=0.2, random_state=42 )
class MLP(nn.Module):
    def __init__(self, input_size):
        super(MLP, self).__init__()
        
        self.model = nn.Sequential(
            nn.Linear(input_size, 128), #learns weight 
            nn.ReLU(), #adds non linearity - negative input values to zero while positive values unchanged
            nn.Dropout(0.3),#prevents overfitting
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(64, 3)  # 3 classes
        )

    def forward(self, x):
        return self.model(x)

input_size = X_train.shape[1]
model = MLP(input_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5) # L2 regularization

epochs = 20#training loop
for epoch in range(epochs):
    
    model.train()
    
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # validation
    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = criterion(val_outputs, y_val)
    
    print(f"Epoch {epoch+1}, Train Loss: {loss.item()}, Val Loss: {val_loss.item()}")

    _, preds = torch.max(val_outputs, 1)
accuracy = (preds == y_val).float().mean()

print("Validation Accuracy:", accuracy.item())

Epoch 1, Train Loss: 124.42622375488281, Val Loss: 7.373736381530762
Epoch 2, Train Loss: 44.72300720214844, Val Loss: 6.930124759674072
Epoch 3, Train Loss: 25.64335060119629, Val Loss: 8.381725311279297
Epoch 4, Train Loss: 10.409282684326172, Val Loss: 11.437492370605469
Epoch 5, Train Loss: 24.177303314208984, Val Loss: 14.577544212341309
Epoch 6, Train Loss: 3.719330072402954, Val Loss: 18.362228393554688
Epoch 7, Train Loss: 3.1371078491210938, Val Loss: 21.886457443237305
Epoch 8, Train Loss: 2.5679194927215576, Val Loss: 24.888919830322266
Epoch 9, Train Loss: 2.729027509689331, Val Loss: 27.486661911010742
Epoch 10, Train Loss: 2.6783339977264404, Val Loss: 29.794513702392578
Epoch 11, Train Loss: 3.262259006500244, Val Loss: 31.769302368164062
Epoch 12, Train Loss: 3.3080689907073975, Val Loss: 33.43521499633789
Epoch 13, Train Loss: 3.20192551612854, Val Loss: 34.772377014160156
Epoch 14, Train Loss: 3.0952088832855225, Val Loss: 35.82793426513672
Epoch 15, Train Loss: 3.598